# Homework 2 — package-based visible marking

Install `homework-marking-system[tables]` in this notebook kernel. Keep
your supplied `retail_data.csv` beside the notebook.
Restart the kernel after installation. Complete the answer cells, then run
the following marking cells. There are 90 visible and 10 manual marks.
Only the teacher changes the marking module or marking cells.

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display
from homework_marking_system.assignments.lab2 import create_marking

# Run once per fresh kernel. Creating a new marking object resets feedback.
marking = create_marking("retail_data.csv")

## Part A – Setup and load (5 marks)
Import pandas and NumPy. Read the supplied CSV into `df` without modifying it.

In [ ]:
### BEGIN SOLUTION
import pandas as pd
import numpy as np
df = pd.read_csv('retail_data.csv')
display(df.head())
### END SOLUTION

In [ ]:
assert isinstance(df, pd.DataFrame)
assert {'Date', 'Region', 'Category', 'Sales', 'Cost', 'Profit'}.issubset(df.columns)

In [ ]:
marking.check('a-loaded-values', df=df)

## Part B – Inspect (10 automatic + 3 manual marks)
Save and display:
 - `first_10` (DataFrame),
 - `data_shape` (tuple),
 - `column_names`(list),
 - `data_types` (Series from `.dtypes`),
 - `numeric_summary`(DataFrame from `.describe()` on numeric columns).

In the written observations cell below, describe at least two data-quality issues (3 manual marks).

In [ ]:
### BEGIN SOLUTION
first_10 = df.head(10)
data_shape = df.shape
column_names = df.columns.tolist()
data_types = df.dtypes
numeric_summary = df.describe()
display(first_10)
print(data_shape, column_names)
display(data_types, numeric_summary)
### END SOLUTION

In [ ]:
marking.check('b-first-ten', first_10=first_10)

In [ ]:
marking.check('b-shape', data_shape=data_shape)

In [ ]:
marking.check('b-column-names', column_names=column_names)

In [ ]:
marking.check('b-dtypes', data_types=data_types)

In [ ]:
marking.check('b-statistics', numeric_summary=numeric_summary)

Region, Category and Cost contain missing values. There are fully duplicated
records. Date is initially a string column rather than a datetime column.
Some Sales values are negative; these need business context (for example,
returns) before deciding whether they are errors.

## Part C – Investigate data quality (15 marks)
Save:
- `missing_counts` (Series indexed by column)
- `missing_rows` (all rows with at least one missing value),
- `duplicate_count` (number of extra fully duplicated rows, keeping the first),
- `duplicate_rows` (those extra rows),
- `type_checks` (dtype Series for Date, Sales, Cost, Profit in that order).

Display the results. Record dtype concerns in Part B's written response.

In [ ]:
### BEGIN SOLUTION
missing_counts = df.isna().sum()
missing_rows = df.loc[df.isna().any(axis=1)]
duplicate_count = int(df.duplicated(keep='first').sum())
duplicate_rows = df.loc[df.duplicated(keep='first')]
type_checks = df[['Date', 'Sales', 'Cost', 'Profit']].dtypes
display(missing_counts, missing_rows, duplicate_rows, type_checks)
print('Extra duplicate rows:', duplicate_count)
### END SOLUTION

In [ ]:
marking.check('c-missing-counts', missing_counts=missing_counts)

In [ ]:
marking.check('c-missing-rows', missing_rows=missing_rows)

In [ ]:
marking.check('c-duplicate-count', duplicate_count=duplicate_count)

In [ ]:
marking.check('c-duplicate-rows', duplicate_rows=duplicate_rows)

In [ ]:
marking.check('c-types', type_checks=type_checks)

## Part D – Clean and prepare (35 automatic + 3 manual marks)
Create `df_clean` as a copy; leave `df` unchanged.
1. Drop fully duplicated rows, keeping the first and retaining row order.
2. Fill missing Region and Category labels with `"Unknown"`.
3. Convert Cost to numeric with invalid values treated as missing, then fill
   missing Cost using its median **after deduplication**.
4. Parse Date using `%d/%m/%Y`.
5. Add Year, Month (1–12), Month_name (English full month name).
6. Display the cleaned shape, dtypes, missing counts and duplicate count.
Explain the median choice and one limitation in the written response.

In [ ]:
### BEGIN SOLUTION
df_clean = df.drop_duplicates(keep='first').copy()
df_clean['Region'] = df_clean['Region'].fillna('Unknown')
df_clean['Category'] = df_clean['Category'].fillna('Unknown')
df_clean['Cost'] = pd.to_numeric(df_clean['Cost'], errors='coerce')
median_cost = df_clean['Cost'].median()
df_clean['Cost'] = df_clean['Cost'].fillna(median_cost)
df_clean['Date'] = pd.to_datetime(df_clean['Date'], format='%d/%m/%Y')
df_clean['Year'] = df_clean['Date'].dt.year
df_clean['Month'] = df_clean['Date'].dt.month
df_clean['Month_name'] = df_clean['Date'].dt.month_name()
print('Shape:', df_clean.shape)
display(df_clean.dtypes, df_clean.isna().sum())
print('Duplicates:', df_clean.duplicated().sum())
print('All dates parsed:', df_clean['Date'].notna().all())
### END SOLUTION

In [ ]:
assert isinstance(df_clean, pd.DataFrame)
assert {'Year', 'Month', 'Month_name'}.issubset(df_clean.columns)

In [ ]:
marking.check('d-original-unchanged', df=df, df_clean=df_clean)

In [ ]:
marking.check('d-deduplication', df_clean=df_clean)

In [ ]:
marking.check('d-region', df_clean=df_clean)

In [ ]:
marking.check('d-category', df_clean=df_clean)

In [ ]:
marking.check('d-cost', df_clean=df_clean)

In [ ]:
marking.check('d-dates', df_clean=df_clean)

In [ ]:
marking.check('d-date-features', df_clean=df_clean)

In [ ]:
marking.check('d-final-quality', df_clean=df_clean)

The median is less sensitive than the mean to unusually large or small costs,
and is a simple way to retain rows with missing Cost. It is calculated after
deduplication so repeated records do not affect the estimate. A limitation is
that a single overall median ignores differences between products/categories
and reduces the observed variation. It does not recover the actual missing cost.

## Part E – Business analysis (15 automatic + 4 manual marks)
Using df_clean, save:
- `regional_summary`: a DataFrame with columns `Region`, `Sales`, `Profit` in
  that order. Region must be an ordinary column, not the index. Sum Sales and
  Profit by Region, then sort by Sales descending.
- `category_summary`: a DataFrame with Category as the index and columns
  `Total_sales`, `Total_profit` in that order (these names are case-sensitive).
  Sum Sales and Profit by Category, then sort by Total_profit descending.

Save `top_region` and `top_category` as the leading group names (strings).
Read `top_region` from the first row's Region column and `top_category`
from the category table's index. Interpret both tables.

In [ ]:
### BEGIN SOLUTION
regional_summary = (
    df_clean.groupby('Region', as_index=False)[['Sales', 'Profit']]
    .sum()
    .sort_values('Sales', ascending=False)
    .reset_index(drop=True)
)
top_region = str(regional_summary.iloc[0]['Region'])
display(regional_summary)
print('Highest sales:', top_region)
### END SOLUTION

In [ ]:
marking.check('e-region-table', regional_summary=regional_summary)

In [ ]:
marking.check('e-region-leader', top_region=top_region)

South has the largest total sales in this dataset. Compare both Sales and
Profit before concluding that the sales leader is also the most profitable.
Unknown is retained as a separate group so missing region labels do not
silently remove revenue from the totals. These are totals, not profit margins.

In [ ]:
### BEGIN SOLUTION
category_summary = (
    df_clean.groupby('Category')
    .agg(Total_sales=('Sales', 'sum'), Total_profit=('Profit', 'sum'))
    .sort_values('Total_profit', ascending=False)
)
top_category = str(category_summary.index[0])
display(category_summary)
print('Highest profit:', top_category)
### END SOLUTION

In [ ]:
marking.check('e-category-table', category_summary=category_summary)

In [ ]:
marking.check('e-category-leader', top_category=top_category)

Office has the highest total profit in this dataset. This identifies the
largest contributor in absolute profit, but does not establish the highest
profit margin or profitability per transaction. Unknown categories are
included in the totals rather than discarded.

## Part F – Monthly sales and export (10 marks)
- Save monthly_sales as a Series of total Sales indexed by Year and Month
(a two-level index), sorted chronologically ascending.

- Export df_clean to
`retail_data_cleaned_homework.csv` with index=False.

- Read the CSV back into
`df_reloaded` using default pd.read_csv settings (dates remain strings).

- Print a confirmation, shape and column names. CSV does not preserve dtypes.

In [ ]:
### BEGIN SOLUTION
monthly_sales = df_clean.groupby(['Year', 'Month'])['Sales'].sum().sort_index()
display(monthly_sales)
### END SOLUTION

In [ ]:
marking.check('f-monthly', monthly_sales=monthly_sales)

In [ ]:
### BEGIN SOLUTION
df_clean.to_csv('retail_data_cleaned_homework.csv', index=False)
df_reloaded = pd.read_csv('retail_data_cleaned_homework.csv')
print('Saved retail_data_cleaned_homework.csv')
print('Reloaded shape:', df_reloaded.shape)
print('Reloaded columns:', df_reloaded.columns.tolist())
assert df_reloaded.shape == df_clean.shape
assert df_reloaded.columns.tolist() == df_clean.columns.tolist()
### END SOLUTION

In [ ]:
marking.check('f-export', df_clean=df_clean)

In [ ]:
marking.check('f-reloaded', df_reloaded=df_reloaded)

## Marking summary
Automatic: A 5 + B 10 + C 15 + D 35 + E 15 + F 10 = **90**.
All 90 automatic marks are visible in this assignment.
Manual: observations 3 + median explanation 3 + interpretations 2 + 2 = **10**.
Each visible test has a unique feedback name and matching nbgrader points.
Rerunning a test replaces its result; it does not add marks twice.
Rerunning the helper cell preserves results. Restart the kernel and Run All
to verify current answers from scratch. Written responses are reviewed manually;
their marks are not awarded by this summary.


In [ ]:
marking.summarise_marks()